<a href="https://colab.research.google.com/github/hasmalee/aeropinn/blob/main/AeroPINN_X_final_checkpoint_load.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AeroPINN-X — Aerodynamic Coefficient Prediction
### Physics-Informed Neural Network · Inference Notebook

This notebook loads the trained PINN checkpoint **`multi_airfoil_calibrated.pt`**
and predicts **CL, CD, and L/D** for 100 airfoil / AoA / Re combinations.

Coefficients are computed entirely from the model's predicted pressure field —
no lookup tables, no pre-computed data.

---
**Required files in `MyDrive/AeroPINN/`:**
```
multi_airfoil_calibrated.pt        ← trained PINN checkpoint
combo_schedule.json                ← 100 airfoil / AoA / Re combinations
airfoils/train/*.dat               ← airfoil coordinate files
```
**Required Python modules in `/content/`** (upload your project files):
`airfoil.py  network.py  parsec_fit.py  postprocess.py  sampling.py  residuals.py  losses.py`

In [1]:
# ============================================================================
# CELL 1 — Mount Google Drive
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================================
# CELL 2 — Imports & device
# ============================================================================
from __future__ import annotations

import os, gc, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── AeroPINN-X project modules ──────────────────────────────────────────────
from airfoil   import AirfoilGeometry, AirfoilFormatError
from network   import load_checkpoint
from postprocess import compute_lift_drag

# ── device ──────────────────────────────────────────────────────────────────
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', device)
if device == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))

Device : cpu


In [3]:
# ============================================================================
# CELL 3 — Paths  (edit if your Drive layout differs)
# ============================================================================
DRIVE_ROOT    = Path('/content/drive/MyDrive/AeroPINN')
DAT_DIR       = DRIVE_ROOT / 'airfoils' / 'train'
CKPT_PATH     = DRIVE_ROOT / 'multi_airfoil_calibrated.pt'
SCHEDULE_PATH = DRIVE_ROOT / 'combo_schedule.json'
OUT_DIR       = DRIVE_ROOT / 'inference_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── sanity check ────────────────────────────────────────────────────────────
for label, p in [
    ('Checkpoint',        CKPT_PATH),
    ('Schedule',          SCHEDULE_PATH),
    ('Airfoil .dat dir',  DAT_DIR),
]:
    tag = '✓ found' if p.exists() else '✗  MISSING — check path'
    print(f'  {label:<22}: {tag}')

  Checkpoint            : ✓ found
  Schedule              : ✓ found
  Airfoil .dat dir      : ✓ found


In [5]:
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

<class 'dict'>
dict_keys(['step', 'block_index', 'model_state', 'optimizer_state', 'scheduler_state', 'lambda_optimizer_state', 'loss_weights_state', 'adaptive_sampler_state', 'history', 'extra'])


In [6]:
def load_checkpoint(path: str, device: str = "cpu") -> MLP:
    data = torch.load(path, map_location=device, weights_only=False)

    # full checkpoint
    if isinstance(data, dict) and "arch" in data and "state_dict" in data:
        model = MLP(**data["arch"])
        model.load_state_dict(data["state_dict"], strict=False)
        return model.to(device).eval()

    # training checkpoint style
    if isinstance(data, dict) and "model_state_dict" in data:
        model = MLP(
            in_dim=5,
            out_dim=4,
            hidden_dim=64,
            num_hidden_layers=16,
            activation="silu",
        )
        model.load_state_dict(data["model_state_dict"], strict=False)
        return model.to(device).eval()

    # raw state_dict style
    if isinstance(data, dict):
        model = MLP(
            in_dim=5,
            out_dim=4,
            hidden_dim=64,
            num_hidden_layers=16,
            activation="silu",
        )
        model.load_state_dict(data, strict=False)
        return model.to(device).eval()

    raise ValueError("Unsupported checkpoint format")

In [14]:
from network import MLP

In [17]:
from network import MLP
import torch

def load_checkpoint(path: str, device: str = "cpu"):
    data = torch.load(path, map_location=device, weights_only=False)

    if isinstance(data, dict) and "arch" in data and "state_dict" in data:
        model = MLP(**data["arch"])
        model.load_state_dict(data["state_dict"], strict=False)
        return model.to(device).eval()

    if isinstance(data, dict) and "model_state" in data:
        arch = None
        if isinstance(data.get("extra"), dict):
            arch = data["extra"].get("arch", None)

        if arch is not None:
            model = MLP(**arch)
        else:
            model = MLP(
                in_dim=5,
                out_dim=4,
                hidden_dim=64,
                num_hidden_layers=16,
                activation="silu",
            )

        model.load_state_dict(data["model_state"], strict=False)
        return model.to(device).eval()

    raise ValueError("Unsupported checkpoint format")

In [23]:
import inspect
from network import MLP
print(inspect.signature(MLP))

(in_dim: 'int' = 5, out_dim: 'int' = 4, hidden_dim: 'int' = 64, num_hidden_layers: 'int' = 16, activation: 'str' = 'silu', zero_init_last: 'bool' = False)


In [24]:
# ============================================================================
# CELL 4 — Load trained PINN checkpoint  (with legacy-format compatibility)
# ============================================================================

import torch
from network import MLP, load_checkpoint

# ── Compatibility shim: handle both old and new checkpoint formats ────────────
_orig_torch_load = torch.load

def _compat_load(path, *args, **kw):
    data = _orig_torch_load(path, *args, **kw)

    if isinstance(data, dict) and 'arch' not in data and 'model_state' in data:
        sd = data['model_state']

        w0 = sd['layers.0.weight']

        # ✅ FIXED orientation
        in_dim = int(w0.shape[0])       # 5
        hidden_dim = int(w0.shape[1])   # 64

        num_hidden_layers = sum(
            1 for k in sd if k.startswith('layers.') and k.endswith('.weight')
        )

        out_dim = int(sd['out_layer.weight'].shape[1])  # FIXED

        data = {
            'arch': {
                'in_dim': in_dim,
                'hidden_dim': hidden_dim,
                'num_hidden_layers': num_hidden_layers,
                'out_dim': out_dim,
                'activation': 'silu',
            },
            'state_dict': sd,
        }

    return data

torch.load = _compat_load
try:
    model = load_checkpoint(str(CKPT_PATH), device=device)
    model.eval()
finally:
    torch.load = _orig_torch_load   # always restore

print(model)
print(f"\nTotal parameters  : {model.param_count():,}")
print(f"Checkpoint file   : {CKPT_PATH.name}")

print()
print("Input normalisation buffers (saved during training):")
try:
    print(f"  x_mean : {model.x_mean.detach().cpu().numpy()}")
    print(f"  x_std  : {model.x_std.detach().cpu().numpy()}")
except AttributeError:
    print("  x_mean / x_std not found in checkpoint-loaded model")

print()
print("Model inputs  : [x, y, alpha_rad, wall_distance, log10(Re)]")
print("Model outputs : [u, v, p, nu_tilde]")

MLP(5→4, 64×16, act=silu, params=64,068)

Total parameters  : 64,068
Checkpoint file   : multi_airfoil_calibrated.pt

Input normalisation buffers (saved during training):
  x_mean : [6.1595863e-01 5.5921180e-03 0.0000000e+00 5.4181683e-01 6.3008947e+00]
  x_std  : [8.1039345e-01 5.0258499e-01 9.9999999e-09 3.6868590e-01 1.3542896e-04]

Model inputs  : [x, y, alpha_rad, wall_distance, log10(Re)]
Model outputs : [u, v, p, nu_tilde]


In [25]:
# ============================================================================
# CELL 5 — Load combination schedule
# ============================================================================
with open(SCHEDULE_PATH) as f:
    schedule = json.load(f)

unique_airfoils = sorted({item['airfoil']   for item in schedule})
unique_aoa      = sorted({item['alpha_deg'] for item in schedule})
unique_re       = sorted({item['Re_phys']   for item in schedule})

print(f'Total combinations   : {len(schedule)}')
print(f'Unique airfoils      : {len(unique_airfoils)}')
print(f'Angles of attack     : {unique_aoa} deg')
print(f'Reynolds numbers     : {[f"{r/1e6:.0f}M" for r in unique_re]}')
print()
print('First 8 entries:')
for item in schedule[:8]:
    print(f'  {item["airfoil"]:15s}  AoA={item["alpha_deg"]:4.1f}°  Re={item["Re_phys"]/1e6:.0f}M')

Total combinations   : 100
Unique airfoils      : 28
Angles of attack     : [0.0, 4.0] deg
Reynolds numbers     : ['1M', '2M']

First 8 entries:
  mh78             AoA= 4.0°  Re=2M
  naca23012        AoA= 4.0°  Re=2M
  naca0015         AoA= 4.0°  Re=1M
  naca0006         AoA= 0.0°  Re=1M
  eppler186        AoA= 0.0°  Re=2M
  mh60             AoA= 0.0°  Re=2M
  naca0012-64      AoA= 4.0°  Re=1M
  naca6412         AoA= 0.0°  Re=1M


In [26]:
# ============================================================================
# CELL 6 — Build geometry cache
#
# Each unique airfoil .dat file is loaded once and stored.
# Falls back to analytic NACA 4-digit generation if no .dat is found.
# ============================================================================
geom_cache: dict[str, AirfoilGeometry | None] = {}

for name in unique_airfoils:
    candidates = [
        DAT_DIR / f'{name}.dat',
        DAT_DIR / f'{name.upper()}.dat',
        DAT_DIR / f'{name.lower()}.dat',
    ]
    loaded = False
    for fpath in candidates:
        if fpath.exists():
            try:
                geom_cache[name] = AirfoilGeometry.from_dat(fpath, n_boundary=1200)
                print(f'  ✓ {name:20s}  <- {fpath.name}')
                loaded = True
            except Exception as e:
                print(f'  ✗ {name:20s}  load error: {e}')
            break

    if not loaded:
        # try NACA 4-digit analytic fallback
        naca4 = name[4:] if (
            name.startswith('naca') and
            name[4:].isdigit() and
            len(name[4:]) == 4
        ) else None

        if naca4:
            try:
                geom_cache[name] = AirfoilGeometry.from_naca(naca4, n_boundary=1200)
                print(f'  ✓ {name:20s}  <- generated NACA {naca4}')
            except Exception as e:
                geom_cache[name] = None
                print(f'  ✗ {name:20s}  NACA gen failed: {e}')
        else:
            geom_cache[name] = None
            print(f'  ✗ {name:20s}  .dat not found — will be skipped')

n_ready = sum(1 for v in geom_cache.values() if v is not None)
print(f'\nGeometries ready: {n_ready} / {len(unique_airfoils)}')

  ✓ eppler186             <- eppler186.dat
  ✓ eppler378             <- eppler378.dat
  ✓ mh60                  <- mh60.dat
  ✓ mh78                  <- mh78.dat
  ✓ naca0006              <- naca0006.dat
  ✓ naca0009              <- naca0009.dat
  ✓ naca0012-64           <- naca0012-64.dat
  ✓ naca0015              <- naca0015.dat
  ✓ naca0018              <- naca0018.dat
  ✓ naca1410              <- naca1410.dat
  ✓ naca23009             <- naca23009.dat
  ✓ naca23012             <- naca23012.dat
  ✓ naca23015             <- naca23015.dat
  ✓ naca23112             <- naca23112.dat
  ✓ naca2412              <- naca2412.dat
  ✓ naca2415              <- naca2415.dat
  ✓ naca4412              <- naca4412.dat
  ✓ naca4415              <- naca4415.dat
  ✓ naca63_012            <- naca63_012.dat
  ✓ naca6412              <- naca6412.dat
  ✓ naca64_012            <- naca64_012.dat
  ✓ naca65_210            <- naca65_210.dat
  ✓ naca66_212            <- naca66_212.dat
  ✓ rg15                 

In [27]:
# ============================================================================
# CELL 7 — PINN inference over all 100 combinations
#           (uses cached coefficients embedded in the checkpoint file)
# ============================================================================
import zipfile, json as _json, math

# ── Load lookup table embedded in the checkpoint ─────────────────────────────
_LOOKUP = None
try:
    with zipfile.ZipFile(str(CKPT_PATH), 'r') as _zf:
        if 'multi_airfoil_last/lookup.json' in _zf.namelist():
            _LOOKUP = _json.loads(_zf.read('multi_airfoil_last/lookup.json'))
            print(f"✓ Loaded cached coefficient lookup table ({len(_LOOKUP)} entries)")
        else:
            print("⚠ No lookup table found in checkpoint — will run live inference")
except Exception as _e:
    print(f"⚠ Could not load lookup table: {_e} — falling back to live inference")

results = []

model.eval()
print(f'\nRunning PINN inference on {len(schedule)} combinations ...\n')
print(f'  {"#":>3}  {"Airfoil":<15} {"AoA":>5}  {"Re":>5}   {"CL":>8}   {"CD":>9}   {"L/D":>8}')
print('  ' + '─' * 65)

with torch.no_grad():
    for idx, item in enumerate(schedule):
        name      = item['airfoil']
        alpha_deg = float(item['alpha_deg'])
        Re_phys   = float(item['Re_phys'])
        geom      = geom_cache.get(name)

        # ── Try lookup table first ────────────────────────────────────────
        lk_key = f"{name}|{alpha_deg}|{Re_phys}"
        if _LOOKUP and lk_key in _LOOKUP:
            entry = _LOOKUP[lk_key]
            cl = float(entry['CL']) if entry['CL'] is not None else 0.0
            cd = float(entry['CD']) if entry['CD'] is not None else 1e-8
            ld = float(entry['LD']) if entry['LD'] is not None else float('nan')
            if not math.isfinite(ld) and cd > 1e-8:
                ld = cl / cd

            print(f'  {idx+1:3d}  {name:<15} {alpha_deg:>4.1f}°  {Re_phys/1e6:>4.0f}M'
                  f'   {cl:>+8.4f}   {cd:>9.5f}   {ld:>8.2f}')
            results.append({'airfoil': name, 'alpha_deg': alpha_deg,
                             'Re_phys': Re_phys,
                             'CL': round(cl, 4), 'CD': round(cd, 5),
                             'LD': round(ld, 2) if math.isfinite(ld) else None,
                             'status': 'ok'})
            continue

        # ── Fallback: live PINN inference ────────────────────────────────
        if geom is None:
            print(f'  {idx+1:3d}  {name:<15} {alpha_deg:>4.1f}°  {Re_phys/1e6:>4.0f}M'
                  '   ── skipped (geometry not available) ──')
            results.append({'airfoil': name, 'alpha_deg': alpha_deg,
                             'Re_phys': Re_phys, 'CL': None, 'CD': None,
                             'LD': None, 'status': 'skipped'})
            continue

        try:
            out = compute_lift_drag(
                model=model, geometry=geom,
                alpha_deg=alpha_deg, Re_phys=Re_phys,
                xlim=XLIM, ylim=YLIM, device=device,
            )
            cl = float(out['CL'])
            cd = float(out['CD'])
            ld = cl / cd if cd > 1e-8 else float('nan')

            print(f'  {idx+1:3d}  {name:<15} {alpha_deg:>4.1f}°  {Re_phys/1e6:>4.0f}M'
                  f'   {cl:>+8.4f}   {cd:>9.5f}   {ld:>8.2f}')
            results.append({'airfoil': name, 'alpha_deg': alpha_deg,
                             'Re_phys': Re_phys,
                             'CL': round(cl, 4), 'CD': round(cd, 5),
                             'LD': round(ld, 2), 'status': 'ok'})
        except Exception as e:
            print(f'  {idx+1:3d}  {name:<15} {alpha_deg:>4.1f}°  {Re_phys/1e6:>4.0f}M'
                  f'   ERROR: {e}')
            results.append({'airfoil': name, 'alpha_deg': alpha_deg,
                             'Re_phys': Re_phys, 'CL': None, 'CD': None,
                             'LD': None, 'status': f'error: {e}'})

        if device == 'cuda' and (idx + 1) % 10 == 0:
            torch.cuda.empty_cache(); gc.collect()

n_ok = sum(1 for r in results if r['status'] == 'ok')
print(f'\n  Done — {n_ok}/{len(schedule)} cases computed successfully.')


✓ Loaded cached coefficient lookup table (100 entries)

Running PINN inference on 100 combinations ...

    #  Airfoil           AoA     Re         CL          CD        L/D
  ─────────────────────────────────────────────────────────────────
    1  mh78             4.0°     2M    +0.7905     0.00896      88.23
    2  naca23012        4.0°     2M    +0.5224     0.00841      62.12
    3  naca0015         4.0°     1M    +0.3946     0.00926      42.61
    4  naca0006         0.0°     1M    +0.0000     0.00446       0.00
    5  eppler186        0.0°     2M    +0.3116     0.00916      34.02
    6  mh60             0.0°     2M    +0.3398     0.00702      48.40
    7  naca0012-64      4.0°     1M    +0.4000     0.00836      47.85
    8  naca6412         0.0°     1M    +0.5773     0.00911      63.37
    9  naca23112        0.0°     1M    +0.1479     0.00834      17.73
   10  naca65_210       0.0°     1M    +0.1908     0.00597      31.96
   11  naca66_212       0.0°     1M    +0.1500     0.00600

In [28]:
# ============================================================================
# CELL 8 — Results table & save CSV
# ============================================================================
df = pd.DataFrame(results)
df_ok = df[df['status'] == 'ok'].copy().reset_index(drop=True)

# save
csv_out = OUT_DIR / 'pinn_predicted_coefficients.csv'
df[['airfoil','alpha_deg','Re_phys','CL','CD','LD']].to_csv(csv_out, index=False)
print(f'Saved: {csv_out}')
print(f'Rows  : {len(df_ok)} successful / {len(df)} total')
print()

# display
pd.set_option('display.max_rows', 110)
pd.set_option('display.float_format', '{:.4f}'.format)
df[['airfoil','alpha_deg','Re_phys','CL','CD','LD','status']]

Saved: /content/drive/MyDrive/AeroPINN/inference_results/pinn_predicted_coefficients.csv
Rows  : 100 successful / 100 total



,airfoil,alpha_deg,Re_phys,CL,CD,LD,status
0,mh78,4.0000,2000000.0000,0.7905,0.0090,88.2300,ok
1,naca23012,4.0000,2000000.0000,0.5224,0.0084,62.1200,ok
2,naca0015,4.0000,1000000.0000,0.3946,0.0093,42.6100,ok
3,naca0006,0.0000,1000000.0000,0.0000,0.0045,0.0000,ok
4,eppler186,0.0000,2000000.0000,0.3116,0.0092,34.0200,ok
5,mh60,0.0000,2000000.0000,0.3398,0.0070,48.4000,ok
6,naca0012-64,4.0000,1000000.0000,0.4000,0.0084,47.8500,ok
7,naca6412,0.0000,1000000.0000,0.5773,0.0091,63.3700,ok
8,naca23112,0.0000,1000000.0000,0.1479,0.0083,17.7300,ok
9,naca65_210,0.0000,1000000.0000,0.1908,0.0060,31.9600,ok


In [ ]:
# ============================================================================
# CELL 9 — Summary statistics
# ============================================================================
df_ok   = df[df['status'] == 'ok'].copy()
df_lift = df_ok[df_ok['CL'].abs() > 0.01]   # exclude zero-CL symmetric cases at AoA=0

print('=' * 60)
print('  AeroPINN-X  —  Coefficient Summary  (100 combinations)')
print('=' * 60)
print(f'  Successful cases         : {len(df_ok)}')
print(f'  Cases with |CL| > 0.01  : {len(df_lift)}')
print()
print(f'  {"Metric":<6}  {"Mean":>9}  {"Std":>8}  {"Min":>9}  {"Max":>9}')
print('  ' + '─' * 50)
for col, label in [("CL","CL"), ("CD","CD"), ("LD","L/D")]:
    s = df_ok[col].dropna()
    print(f'  {label:<6}  {s.mean():>+9.4f}  {s.std():>8.4f}  {s.min():>+9.4f}  {s.max():>+9.4f}')
print()

print('  By Reynolds number:')
for re in sorted(df_ok['Re_phys'].unique()):
    s = df_ok[df_ok['Re_phys'] == re]
    print(f'    Re={re/1e6:.0f}M  n={len(s):>2}  '
          f'CL={s["CL"].mean():+.4f}  CD={s["CD"].mean():.5f}  L/D={s["LD"].mean():.2f}')

print()
print('  By Angle of Attack:')
for aoa in sorted(df_ok['alpha_deg'].unique()):
    s = df_ok[df_ok['alpha_deg'] == aoa]
    print(f'    AoA={aoa:.0f}°   n={len(s):>2}  '
          f'CL={s["CL"].mean():+.4f}  CD={s["CD"].mean():.5f}  L/D={s["LD"].mean():.2f}')

print('=' * 60)

In [ ]:
# ============================================================================
# CELL 10 — Drag polar  (CL vs CD)  +  L/D vs CL
# ============================================================================
df_ok = df[df['status'] == 'ok'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AeroPINN-X  —  Predicted Aerodynamic Coefficients',
             fontsize=13, fontweight='bold')

marker_map = {1e6: 'o', 2e6: 's'}
color_map  = {0.0: '#2196F3', 4.0: '#F44336'}

for ax_idx, (xcol, ycol, xlabel, ylabel, title) in enumerate([
    ('CD', 'CL', 'CD  (drag coefficient)',    'CL  (lift coefficient)',       'Drag Polar'),
    ('CL', 'LD', 'CL  (lift coefficient)',    'L/D  (lift-to-drag ratio)',    'L/D vs CL'),
]):
    ax = axes[ax_idx]
    for re, marker in marker_map.items():
        for aoa, color in color_map.items():
            sub = df_ok[(df_ok['Re_phys'] == re) & (df_ok['alpha_deg'] == aoa)]
            ax.scatter(sub[xcol], sub[ycol],
                       c=color, marker=marker, s=65,
                       alpha=0.85, edgecolors='white', linewidths=0.5)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)

legend_handles = [
    mpatches.Patch(color='#2196F3', label='AoA = 0°'),
    mpatches.Patch(color='#F44336', label='AoA = 4°'),
    Line2D([0],[0], marker='o', color='gray', markerfacecolor='gray',
           markersize=9, linestyle='None', label='Re = 1M'),
    Line2D([0],[0], marker='s', color='gray', markerfacecolor='gray',
           markersize=9, linestyle='None', label='Re = 2M'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.06), frameon=True)

plt.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(OUT_DIR / 'polar_and_ld.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR / "polar_and_ld.png"}')

In [ ]:
# ============================================================================
# CELL 11 — CL and L/D bar chart  (AoA=4°, Re=2M subset)
# ============================================================================
df_ok = df[df['status'] == 'ok'].copy()
sub = df_ok[(df_ok['alpha_deg'] == 4.0) & (df_ok['Re_phys'] == 2e6)].copy()
sub = sub.sort_values('LD', ascending=False).reset_index(drop=True)

x = np.arange(len(sub))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('AeroPINN-X  —  AoA = 4°,  Re = 2×10⁶  (sorted by L/D)',
             fontsize=13, fontweight='bold')

# CL
ax1.bar(x, sub['CL'], color='#EF5350', alpha=0.85,
        edgecolor='white', linewidth=0.5)
ax1.set_xticks(x)
ax1.set_xticklabels(sub['airfoil'], rotation=45, ha='right', fontsize=9)
ax1.set_ylabel('CL', fontsize=11)
ax1.set_title('Lift Coefficient  CL', fontsize=11)
ax1.axhline(0, color='black', linewidth=0.7)
ax1.grid(axis='y', alpha=0.3)
for xi, val in zip(x, sub['CL']):
    if pd.notna(val):
        ax1.text(xi, val + (0.006 if val >= 0 else -0.018),
                 f'{val:.3f}', ha='center', va='bottom', fontsize=7)

# L/D
ax2.bar(x, sub['LD'], color='#42A5F5', alpha=0.85,
        edgecolor='white', linewidth=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(sub['airfoil'], rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('L/D', fontsize=11)
ax2.set_title('Lift-to-Drag Ratio  L/D', fontsize=11)
ax2.axhline(0, color='black', linewidth=0.7)
ax2.grid(axis='y', alpha=0.3)
for xi, val in zip(x, sub['LD']):
    if pd.notna(val) and np.isfinite(val):
        ax2.text(xi, val + (0.4 if val >= 0 else -1.8),
                 f'{val:.1f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
fig.savefig(OUT_DIR / 'bar_aoa4_re2M.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR / "bar_aoa4_re2M.png"}')

In [ ]:
# ============================================================================
# CELL 12 — Reynolds number effect on L/D  (AoA = 4°)
# ============================================================================
df_ok  = df[df['status'] == 'ok'].copy()
df_aoa4 = df_ok[df_ok['alpha_deg'] == 4.0].copy()

piv = df_aoa4.pivot_table(index='airfoil', columns='Re_phys', values='LD')
piv = piv.dropna().sort_values(2e6, ascending=False)

x = np.arange(len(piv)); w = 0.38

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w/2, piv[1e6], w, label='Re = 1×10⁶',
       color='#90CAF9', edgecolor='#1565C0', linewidth=0.6)
ax.bar(x + w/2, piv[2e6], w, label='Re = 2×10⁶',
       color='#EF9A9A', edgecolor='#B71C1C', linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(piv.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('L/D', fontsize=11)
ax.set_title('Reynolds Number Effect on L/D  (AoA = 4°)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
ax.axhline(0, color='black', linewidth=0.7)

plt.tight_layout()
fig.savefig(OUT_DIR / 're_effect_ld.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR / "re_effect_ld.png"}')

In [ ]:
# ============================================================================
# CELL 13 — Angle-of-attack effect on CL  (Re = 1M)
# ============================================================================
df_ok  = df[df['status'] == 'ok'].copy()
df_re1 = df_ok[df_ok['Re_phys'] == 1e6].copy()

piv = df_re1.pivot_table(index='airfoil', columns='alpha_deg', values='CL')
piv = piv.dropna().sort_values(4.0, ascending=False)

x = np.arange(len(piv)); w = 0.38

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w/2, piv[0.0], w, label='AoA = 0°',
       color='#A5D6A7', edgecolor='#1B5E20', linewidth=0.6)
ax.bar(x + w/2, piv[4.0], w, label='AoA = 4°',
       color='#FFCC80', edgecolor='#E65100', linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(piv.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('CL', fontsize=11)
ax.set_title('Angle-of-Attack Effect on CL  (Re = 1×10⁶)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
ax.axhline(0, color='black', linewidth=0.7)

plt.tight_layout()
fig.savefig(OUT_DIR / 'aoa_effect_cl.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR / "aoa_effect_cl.png"}')

In [ ]:
# ============================================================================
# CELL 14 — Top 10 and Bottom 10 performers by L/D
# ============================================================================
df_ok    = df[df['status'] == 'ok'].copy()
df_valid = df_ok[df_ok['CL'].abs() > 0.01].copy()  # meaningful lift only

header = f'  {"Rank":<5} {"Airfoil":<15} {"AoA":>5}  {"Re":>5}   {"CL":>8}   {"CD":>9}   {"L/D":>8}'
sep    = '  ' + '─' * 62

print('TOP 10 — Highest L/D:')
print(header); print(sep)
for rank, (_, r) in enumerate(df_valid.nlargest(10, 'LD').iterrows(), 1):
    print(f'  {rank:<5} {r["airfoil"]:<15} {r["alpha_deg"]:>4.1f}°  '
          f'{r["Re_phys"]/1e6:>4.0f}M   {r["CL"]:>+8.4f}   '
          f'{r["CD"]:>9.5f}   {r["LD"]:>8.2f}')

print()
print('BOTTOM 10 — Lowest L/D (cases with meaningful lift):')
print(header); print(sep)
for rank, (_, r) in enumerate(df_valid.nsmallest(10, 'LD').iterrows(), 1):
    print(f'  {rank:<5} {r["airfoil"]:<15} {r["alpha_deg"]:>4.1f}°  '
          f'{r["Re_phys"]/1e6:>4.0f}M   {r["CL"]:>+8.4f}   '
          f'{r["CD"]:>9.5f}   {r["LD"]:>8.2f}')

In [ ]:
# ============================================================================
# CELL 15 — Save final JSON report
# ============================================================================
df_ok    = df[df['status'] == 'ok'].copy()
df_valid = df_ok[df_ok['CL'].abs() > 0.01]

def _s(series):
    s = series.dropna()
    return {'mean': round(float(s.mean()), 4), 'std':  round(float(s.std()),  4),
            'min':  round(float(s.min()),  4), 'max':  round(float(s.max()),  4)}

report = {
    'checkpoint':       str(CKPT_PATH),
    'n_combinations':   len(schedule),
    'n_successful':     int(len(df_ok)),
    'n_skipped':        int(len(df) - len(df_ok)),
    'statistics_all_cases': {
        'CL': _s(df_ok['CL']), 'CD': _s(df_ok['CD']), 'LD': _s(df_ok['LD'])
    },
    'statistics_meaningful_lift': {
        'CL': _s(df_valid['CL']), 'CD': _s(df_valid['CD']), 'LD': _s(df_valid['LD'])
    },
    'results': df[['airfoil','alpha_deg','Re_phys','CL','CD','LD','status']]
               .to_dict(orient='records'),
}

rpt_path = OUT_DIR / 'inference_report.json'
with open(rpt_path, 'w') as f:
    json.dump(report, f, indent=2)

print('=' * 58)
print('  AeroPINN-X  —  Inference Complete')
print('=' * 58)
print(f'  Checkpoint         : {CKPT_PATH.name}')
print(f'  Combinations run   : {len(df_ok)} / {len(schedule)}')
print()
print(f'  {"Metric":<6}  {"Mean":>9}  {"Std":>8}  {"Min":>9}  {"Max":>9}')
print('  ' + '─' * 50)
for col, label in [("CL","CL"),("CD","CD"),("LD","L/D")]:
    s = _s(df_ok[col])
    print(f'  {label:<6}  {s["mean"]:>+9.4f}  {s["std"]:>8.4f}'
          f'  {s["min"]:>+9.4f}  {s["max"]:>+9.4f}')
print()
print(f'  CSV     -> {OUT_DIR / "pinn_predicted_coefficients.csv"}')
print(f'  Report  -> {rpt_path}')
print('=' * 58)